In [1]:
# A100 train
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds = ds.train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 5372
    })
})

In [4]:
ds['train'][:3]

{'output': ['截止到目前为止是2021年，这个人将满21岁。当然，具体年龄还需要根据当前日期来确定，譬如：如果今天是2021年8月9日或之后，那么这个人应该是21岁。如果今天是2021年8月8日或之前，则这个人应该是20岁。',
  '对不起，您并未提供任何具体的引语。请您提供一段具体的引语，这样我才能根据它的内容和语境为您将其归类为悲观或乐观。',
  '"探索无限，挑战不可能：SpaceX让我们一起登上未来之星!"'],
 'input': ['', '', ''],
 'instruction': ['计算一个出生于2000年8月9日的人的年龄。', '将此引语归类为悲观或乐观。', '为SpaceX创建口号。']}

In [5]:
tokenizer = AutoTokenizer.from_pretrained("/tmp/code/chatglm3-6b", trust_remote_code=True)
tokenizer

ChatGLMTokenizer(name_or_path='/tmp/code/chatglm3-6b', vocab_size=64798, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='left', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	
}
)

In [6]:
tokenizer(tokenizer.eos_token), tokenizer.eos_token_id

({'input_ids': [64790, 64792, 2893, 30917, 30994], 'attention_mask': [1, 1, 1, 1, 1], 'position_ids': [0, 1, 2, 3, 4]},
 2)

In [7]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = "\n".join([example["instruction"], example["input"]]).strip()     # query
    instruction = tokenizer.build_chat_input(instruction, history=[], role="user")  # [gMASK]sop<|user|> \n query<|assistant|>
    response = tokenizer("\n" + example["output"], add_special_tokens=False)        # \n response, 缺少eos token
    input_ids = instruction["input_ids"][0].numpy().tolist() + response["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = instruction["attention_mask"][0].numpy().tolist() + response["attention_mask"] + [1]
    labels = [-100] * len(instruction["input_ids"][0].numpy().tolist()) + response["input_ids"] + [tokenizer.eos_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

Map: 100%|██████████| 5372/5372 [00:02<00:00, 1842.40 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5372
    })
})

In [9]:
tokenizer.decode(tokenized_ds['train'][1]["input_ids"])

'[gMASK]sop<|user|> \n 将此引语归类为悲观或乐观。<|assistant|> \n对不起，您并未提供任何具体的引语。请您提供一段具体的引语，这样我才能根据它的内容和语境为您将其归类为悲观或乐观。'

In [10]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds['train'][1]["labels"])))

'\n对不起，您并未提供任何具体的引语。请您提供一段具体的引语，这样我才能根据它的内容和语境为您将其归类为悲观或乐观。'

In [12]:
import torch
model = AutoModelForCausalLM.from_pretrained("/tmp/code/chatglm3-6b", 
                                             trust_remote_code=True, 
                                             low_cpu_mem_usage=True, 
                                             torch_dtype=torch.bfloat16, load_in_8bit=True)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 7/7 [00:28<00:00,  4.05s/it]


In [13]:
for name, param in model.named_parameters():
    print(name, param.dtype)

transformer.embedding.word_embeddings.weight torch.bfloat16
transformer.encoder.layers.0.input_layernorm.weight torch.bfloat16
transformer.encoder.layers.0.self_attention.query_key_value.weight torch.int8
transformer.encoder.layers.0.self_attention.query_key_value.bias torch.bfloat16
transformer.encoder.layers.0.self_attention.dense.weight torch.int8
transformer.encoder.layers.0.post_attention_layernorm.weight torch.bfloat16
transformer.encoder.layers.0.mlp.dense_h_to_4h.weight torch.int8
transformer.encoder.layers.0.mlp.dense_4h_to_h.weight torch.int8
transformer.encoder.layers.1.input_layernorm.weight torch.bfloat16
transformer.encoder.layers.1.self_attention.query_key_value.weight torch.int8
transformer.encoder.layers.1.self_attention.query_key_value.bias torch.bfloat16
transformer.encoder.layers.1.self_attention.dense.weight torch.int8
transformer.encoder.layers.1.post_attention_layernorm.weight torch.bfloat16
transformer.encoder.layers.1.mlp.dense_h_to_4h.weight torch.int8
transfo

In [14]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

config = LoraConfig(target_modules=["query_key_value"])
config

LoraConfig(task_type=None, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'query_key_value'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

In [15]:
model = get_peft_model(model, config)

In [16]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.transformer.embedding.word_embeddings.weight
base_model.model.transformer.encoder.layers.0.input_layernorm.weight
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.base_layer.weight
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.base_layer.bias
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.lora_A.default.weight
base_model.model.transformer.encoder.layers.0.self_attention.query_key_value.lora_B.default.weight
base_model.model.transformer.encoder.layers.0.self_attention.dense.weight
base_model.model.transformer.encoder.layers.0.post_attention_layernorm.weight
base_model.model.transformer.encoder.layers.0.mlp.dense_h_to_4h.weight
base_model.model.transformer.encoder.layers.0.mlp.dense_4h_to_h.weight
base_model.model.transformer.encoder.layers.1.input_layernorm.weight
base_model.model.transformer.encoder.layers.1.self_attention.query_key_value.base_layer.weight
base_model.model.transfor

In [17]:
model.print_trainable_parameters()

trainable params: 1,949,696 || all params: 6,245,533,696 || trainable%: 0.0312


In [18]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    logging_steps=50,
    eval_strategy='steps',
    num_train_epochs=1,
    learning_rate=1e-4,
    remove_unused_columns=False
)

In [21]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds['train'].select(range(10000)),
    eval_dataset=tokenized_ds['test'].select(range(2000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

/tmp/ipykernel_266/3768420791.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [22]:
trainer.train()

Step,Training Loss,Validation Loss
50,2.329500,2.073375
100,2.047900,1.945187
150,1.926800,1.904125
200,1.848900,1.890250
250,1.894200,1.884437
300,1.869800,1.881375


TrainOutput(global_step=313, training_loss=1.9856885233626198, metrics={'train_runtime': 654.2937, 'train_samples_per_second': 15.284, 'train_steps_per_second': 0.478, 'total_flos': 5.084746028207309e+16, 'train_loss': 1.9856885233626198, 'epoch': 1.0})

In [23]:
model.eval()
print(model.chat(tokenizer, "数学考试怎么考高分？", history=[])[0])

/usr/local/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


数学考试取得高分，需要充分准备，掌握数学的基本概念和公式，并能够熟练运用。以下是一些建议，帮助你在数学考试中取得高分：

1. 充分准备：要花时间复习和练习数学，了解考试的内容和形式。可以参考教材和辅导资料，并做一些练习题。

2. 理解概念：不要只满足于死记硬背，而要理解数学概念，了解其应用。

3. 练习技巧：练习做题的方法和技巧，并注意解题方法的优化和策略的运用。

4. 做题方法：要掌握解题技巧，学会分析问题，找到关键信息，并灵活运用公式和定理。

5. 做题速度：做题时要注意速度和准确性的平衡，不要因过于追求速度而影响准确性。

6. 考试策略：在考试中，可以先解答自己熟悉的题目，然后再处理相对困难的题目，并合理分配时间。

7. 保持冷静：在考试过程中，保持冷静和自信，不要被难题吓倒，相信自己能够完成考试。

8. 复习策略：定期进行复习，巩固学过的知识，并检查自己是否掌握了数学的基本概念和公式。

9. 寻求帮助：遇到问题可以向老师、同学或家长寻求帮助，并多与他们交流，共同提高数学水平。

10. 保持积极态度：积极面对数学考试，相信自己能够取得好成绩。

通过以上这些方法，你可以提高数学考试的成绩。同时，不断练习和学习，积累经验，你会越来越擅长数学。
